In [0]:
from pyspark.sql.functions import current_timestamp, md5, concat_ws, col
from pyspark.sql import DataFrame

def add_audit_columns(df: DataFrame) -> DataFrame:
    """
    Appends standard audit timestamps to track when a record was processed.
    """
    return df.withColumn("_inserted_ts", current_timestamp()) \
             .withColumn("_updated_ts", current_timestamp())

def generate_surrogate_key(df: DataFrame, key_col_name: str, source_cols: list) -> DataFrame:
    """
    Generates a deterministic MD5 hash surrogate key from a list of columns.
    """
    # Pro-Tip: We use '||' as a separator in concat_ws to prevent accidental hash 
    # collisions between columns (e.g., col1="A", col2="BC" vs col1="AB", col2="C")
    return df.withColumn(
        key_col_name, 
        md5(concat_ws("||", *[col(c) for c in source_cols]))
    )